# Task 2.1: Dataset Selection and Setup
## Online Multiscale Dynamic Topic Models (MDTM)

**Roll Number**: 230079 | **Name**: Mehak Jain

---


## Dataset Choice: Synthetic Temporal Document Corpus

We use a **synthetic Dirichlet-Multinomial document corpus** where topics evolve across discrete time epochs. The dataset is generated by sampling documents from a known set of topics whose word distributions shift over time according to a predefined schedule. This is a reasonable testbed for MDTM because: (1) it is a document corpus over time, matching the problem type addressed by the paper — sequential topic modelling; (2) we have full control over the ground-truth topic evolution patterns, allowing us to verify whether the model correctly recovers the dynamics; (3) the synthetic setup lets us cleanly test the multiscale aspect by injecting both slow-evolving and fast-evolving topics.

**Limitations** compared to the original paper's dataset: The paper evaluates on real-world corpora such as NIPS papers (1987–1999) and PNAS scientific articles sorted by publication year. Our synthetic data has a much smaller vocabulary, fewer documents per epoch, and lacks the organic complexity of real text (polysemy, syntax, rare words). However, these limitations are acceptable for a toy reproduction because the focus is on demonstrating the algorithmic mechanism, not achieving state-of-the-art perplexity on a benchmark.


## Preprocessing Steps
1. **Vocabulary**: A fixed vocabulary of 50 words is used.
2. **Topics**: 3 ground-truth topics, each defined by a distribution over the 50 words.
3. **Epochs**: 20 time epochs. Topic 1 evolves slowly, Topic 2 has a medium rate, Topic 3 changes abruptly at epoch 10.
4. **Documents per epoch**: 50 documents, each with approximately 30–60 words.
5. **Random seed**: Fixed at 42 for reproducibility.


In [1]:
# ---- Configuration and Random Seed ----
import numpy as np
import os

SEED = 42
np.random.seed(SEED)

# --- Hyperparameters ---
V = 50          # Vocabulary size
K = 3           # Number of topics
T = 20          # Number of epochs
D_per_epoch = 50  # Documents per epoch
avg_doc_len = 45  # Average document length
S = 3           # Number of timescales for MDTM

print(f"Vocabulary size: {V}")
print(f"Topics: {K}")
print(f"Epochs: {T}")
print(f"Documents per epoch: {D_per_epoch}")
print(f"Total documents: {T * D_per_epoch}")


Vocabulary size: 50
Topics: 3
Epochs: 20
Documents per epoch: 50
Total documents: 1000


The cell above sets global configuration parameters. These correspond to the model parameters described in Section 3.1 of the MDTM paper: $V$ (vocabulary size), $K$ (number of topics), $T$ (number of epochs), $S$ (number of timescales).


In [2]:
# ---- Generate Ground-Truth Evolving Topic-Word Distributions ----

def generate_evolving_topics(V, K, T):
    """Generate ground-truth topic-word distributions that evolve over T epochs.
    
    Topic 0: Slowly evolving (long-timescale behaviour)
    Topic 1: Medium evolution rate
    Topic 2: Abrupt shift at epoch 10 (tests short-timescale adaptation)
    """
    topics = np.zeros((T, K, V))
    
    # Base distributions (Dirichlet draws)
    rng = np.random.RandomState(SEED)
    base_topics = rng.dirichlet(np.ones(V) * 0.3, size=K)
    alt_topic_2 = rng.dirichlet(np.ones(V) * 0.3)  # Alternative for Topic 2's shift
    
    for t in range(T):
        for k in range(K):
            if k == 0:
                # Slow drift: small perturbation each epoch
                noise = rng.dirichlet(np.ones(V) * 200)
                topics[t, k] = 0.95 * base_topics[k] + 0.05 * noise
            elif k == 1:
                # Medium drift
                noise = rng.dirichlet(np.ones(V) * 50)
                topics[t, k] = 0.85 * base_topics[k] + 0.15 * noise
            else:
                # Abrupt shift at epoch 10
                if t < 10:
                    noise = rng.dirichlet(np.ones(V) * 200)
                    topics[t, k] = 0.95 * base_topics[k] + 0.05 * noise
                else:
                    noise = rng.dirichlet(np.ones(V) * 200)
                    topics[t, k] = 0.95 * alt_topic_2 + 0.05 * noise
            
            # Normalize
            topics[t, k] /= topics[t, k].sum()
    
    return topics

true_topics = generate_evolving_topics(V, K, T)
print(f"Ground-truth topics shape: {true_topics.shape}  (epochs x topics x vocab)")
print(f"Topic 0 drift (epoch 0 vs 19): {np.sum(np.abs(true_topics[0,0] - true_topics[19,0])):.4f}")
print(f"Topic 2 drift (epoch 9 vs 10): {np.sum(np.abs(true_topics[9,2] - true_topics[10,2])):.4f}")


Ground-truth topics shape: (20, 3, 50)  (epochs x topics x vocab)
Topic 0 drift (epoch 0 vs 19): 0.0037
Topic 2 drift (epoch 9 vs 10): 1.1416


The cell above generates ground-truth topic-word distributions that evolve according to three different patterns. This is not part of the MDTM algorithm itself; it simulates the real-world phenomenon that MDTM is designed to capture (Section 2 of the paper: "topics in the real world evolve at different timescales").


In [3]:
# ---- Generate Synthetic Documents ----

def generate_documents(true_topics, D_per_epoch, avg_doc_len, alpha=1.0):
    """Generate documents from the evolving topic model.
    
    For each epoch: 
      - Draw topic proportions for each doc from Dirichlet(alpha)
      - Draw words from corresponding topic-word distributions
    
    Returns:
        corpus: list of T lists, each containing D_per_epoch documents
                each document is a list of word indices
        theta_true: (T, D_per_epoch, K) true topic proportions
    """
    T, K, V = true_topics.shape
    rng = np.random.RandomState(SEED + 1)
    corpus = []
    theta_true = np.zeros((T, D_per_epoch, K))
    
    for t in range(T):
        epoch_docs = []
        for d in range(D_per_epoch):
            # Draw topic proportions
            theta = rng.dirichlet(np.ones(K) * alpha)
            theta_true[t, d] = theta
            
            # Draw document length
            doc_len = rng.poisson(avg_doc_len) + 10
            
            # Draw words
            doc = []
            for _ in range(doc_len):
                z = rng.choice(K, p=theta)  # Draw topic
                w = rng.choice(V, p=true_topics[t, z])  # Draw word
                doc.append(w)
            epoch_docs.append(doc)
        corpus.append(epoch_docs)
    
    return corpus, theta_true

corpus, theta_true = generate_documents(true_topics, D_per_epoch, avg_doc_len)
print(f"Corpus: {len(corpus)} epochs, {len(corpus[0])} docs/epoch")
print(f"Example doc length: {len(corpus[0][0])} words")
print(f"Sample words from epoch 0, doc 0: {corpus[0][0][:10]}")


Corpus: 20 epochs, 50 docs/epoch
Example doc length: 49 words
Sample words from epoch 0, doc 0: [33, 2, 16, 9, 42, 18, 2, 38, 13, 15]


The cell above generates synthetic documents following the LDA generative process described in Section 3.1 of the paper. Each document draws topic proportions $\theta_d \sim \text{Dirichlet}(\alpha)$, then for each word position draws a topic $z \sim \text{Multinomial}(\theta_d)$ and a word $w \sim \text{Multinomial}(\phi_z^{(t)})$.


In [4]:
# ---- Save Dataset ----

# Save corpus as numpy arrays for reproducibility
os.makedirs('data', exist_ok=True)

# Convert to word-count matrix per epoch
word_counts = np.zeros((T, D_per_epoch, V), dtype=np.int32)
for t in range(T):
    for d in range(D_per_epoch):
        for w in corpus[t][d]:
            word_counts[t, d, w] += 1

np.save('data/word_counts.npy', word_counts)
np.save('data/true_topics.npy', true_topics)
np.save('data/theta_true.npy', theta_true)

print(f"Saved word_counts.npy: shape {word_counts.shape}")
print(f"Saved true_topics.npy: shape {true_topics.shape}")
print(f"Saved theta_true.npy: shape {theta_true.shape}")
print(f"Total word tokens: {word_counts.sum()}")
print(f"Min doc length: {word_counts.sum(axis=2).min()}")
print(f"Max doc length: {word_counts.sum(axis=2).max()}")


Saved word_counts.npy: shape (20, 50, 50)
Saved true_topics.npy: shape (20, 3, 50)
Saved theta_true.npy: shape (20, 50, 3)
Total word tokens: 54858
Min doc length: 34
Max doc length: 82


The cell above converts the raw document word lists into word-count matrices (bag-of-words representation) and saves them. The bag-of-words assumption is fundamental to the LDA family of models, including MDTM (Section 3.1).
